[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhiyunli/cpsc5610-labs/blob/main/week9/lab_autoencoders.ipynb)

# Week 9 Lab: Autoencoders, from PCA to the VAE

**CPSC 5610 — Deep Learning**

In this lab you build the whole autoencoder family on **Fashion-MNIST**, one variant at a time, and watch what each constraint buys you. The scaffolding (data, training loop, plots, scoreboard) is written for you — you fill in **6 short `TODO`s**, run the cells, and compare the results. Each `TODO` prints a ✓ when it is right.

**You will build and compare:**

1. an **undercomplete linear AE** — and check it matches **PCA**;
2. a **stacked (deep) AE**;
3. a **convolutional AE**;
4. a **denoising AE** (corruption instead of a bottleneck);
5. a **variational AE (VAE)** — then *generate* brand-new garments.

A GPU helps but is optional.

## Part 0 — Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 5610
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## Part 1 — The data and shared helpers

We load Fashion-MNIST and define a training loop, a reconstruction plotter, and a `scoreboard` so every model is measured the same way.

In [ ]:
tfm = transforms.ToTensor()                 # images -> float in [0, 1], shape (1, 28, 28)
train_ds = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=tfm)
test_ds  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)
CLASSES = ['T-shirt','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']

# a fixed batch of test images, so every "before/after" plot is comparable
fixed_x, fixed_y = next(iter(DataLoader(test_ds, batch_size=8, shuffle=False)))

def show_row(imgs, title=None):
    imgs = imgs.detach().cpu()
    fig, axs = plt.subplots(1, len(imgs), figsize=(1.4 * len(imgs), 1.7))
    for ax, im in zip(axs, imgs):
        ax.imshow(im.squeeze(), cmap='gray'); ax.axis('off')
    if title: fig.suptitle(title, fontsize=11)
    plt.show()

def show_recon(model, x=fixed_x, title='reconstructions', vae=False):
    model.eval()
    with torch.no_grad():
        out = model(x.to(device)); out = out[0] if vae else out
    fig, axs = plt.subplots(2, len(x), figsize=(1.4 * len(x), 3))
    for k in range(len(x)):
        axs[0, k].imshow(x[k].squeeze(), cmap='gray'); axs[0, k].axis('off')
        axs[1, k].imshow(out[k].detach().cpu().squeeze(), cmap='gray'); axs[1, k].axis('off')
    fig.suptitle(title + '   (top: input,  bottom: reconstruction)'); plt.show()

@torch.no_grad()
def recon_mse(model, loader, vae=False):
    model.eval(); se, n = 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        out = model(x); out = out[0] if vae else out
        se += F.mse_loss(out, x, reduction='sum').item(); n += x.numel()
    return se / n

def train(model, loader, epochs=10, lr=1e-3, loss_fn=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    if loss_fn is None:
        loss_fn = lambda out, x: F.mse_loss(out, x)
    for ep in range(epochs):
        model.train(); tot, n = 0.0, 0
        for x, _ in loader:
            x = x.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), x)
            loss.backward(); opt.step()
            tot += loss.item() * x.size(0); n += x.size(0)
        if (ep + 1) % max(1, epochs // 3) == 0 or ep == epochs - 1:
            print(f'  epoch {ep + 1:02d}  loss {tot / n:.4f}')

scoreboard = {}
def score(name, model, vae=False):
    m = recon_mse(model, test_loader, vae=vae)
    scoreboard[name] = m
    print(f'{name:<14} test recon MSE = {m:.5f}')
    return m

show_row(fixed_x, 'a few Fashion-MNIST items')

## Part 2 — Undercomplete **linear** AE  (= PCA)

No activations + MSE loss means the AE can only learn a linear projection — the same subspace PCA finds. We train it, then compare its reconstruction error to PCA with the same number of components.

<img src='lab9_slides/ae_slide07_linear_is_pca.png' width='640'>

### TODO 1 — build the linear AE

In [ ]:
K_LIN = 32

class LinearAE(nn.Module):
    def __init__(self, k=K_LIN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, k))
        self.decoder = nn.Sequential(nn.Linear(k, 28 * 28), nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
lin = LinearAE().to(device)
assert lin(fixed_x.to(device)).shape == fixed_x.shape, "LinearAE output must be (B, 1, 28, 28)"
print("✓ TODO 1 check passed.")
train(lin, train_loader, epochs=5, lr=1e-3)
score('linear AE', lin)
show_recon(lin, title='Linear AE')

# A linear AE trained on MSE learns the same subspace as PCA -- compare reconstruction error.
from sklearn.decomposition import PCA
Xtr = train_ds.data.numpy().reshape(len(train_ds), -1).astype('float32') / 255.0
Xte = test_ds.data.numpy().reshape(len(test_ds), -1).astype('float32') / 255.0
pca = PCA(n_components=K_LIN).fit(Xtr)
rec = pca.inverse_transform(pca.transform(Xte))
print(f'PCA({K_LIN}) test recon MSE = {((rec - Xte) ** 2).mean():.5f}   <- the linear AE should land near this')

## Part 3 — Stacked (deep) AE

Add hidden layers and nonlinearities. The decoder mirrors the encoder.

<img src='lab9_slides/ae_slide10_stacked_concept.png' width='640'>

### TODO 2 — build the mirror decoder

In [ ]:
class StackedAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 32), nn.ReLU())
        self.decoder = nn.Sequential(
            nn.Linear(32, 128), nn.ReLU(),
            nn.Linear(128, 28 * 28), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
stacked = StackedAE().to(device)
assert stacked(fixed_x.to(device)).shape == fixed_x.shape, "StackedAE output must be (B, 1, 28, 28)"
print("✓ TODO 2 check passed.")
train(stacked, train_loader, epochs=10)
score('stacked AE', stacked)
show_recon(stacked, title='Stacked AE')

## Part 4 — Convolutional AE

Convolutions exploit image structure. The encoder downsamples with stride-2 convs (28→14→7); the decoder upsamples with `ConvTranspose2d`.

<img src='lab9_slides/ae_slide20_conv_arch.png' width='640'>

### TODO 3 — build the ConvTranspose decoder

In [ ]:
class ConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),    # -> 16 x 14 x 14
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU())   # -> 32 x 7 x 7
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),  # -> 16 x 14 x 14
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid()) # -> 1 x 28 x 28
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
conv = ConvAE().to(device)
assert conv(fixed_x.to(device)).shape == fixed_x.shape, "ConvAE output must be (B, 1, 28, 28)"
print("✓ TODO 3 check passed.")
train(conv, train_loader, epochs=8)
score('conv AE', conv)
show_recon(conv, title='Convolutional AE')

## Part 5 — Denoising AE

Corrupt the input, ask for the clean image back. A `Dropout` at the very front does the corruption; the loss target stays clean. The code can stay wide — the noise is what stops trivial copying.

<img src='lab9_slides/ae_slide22_denoise_idea.png' width='640'>

### TODO 4 — add the corruption

In [ ]:
class DenoisingAE(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p),                       # corrupts the input (training only)
            nn.Linear(28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU())      # code stays wide -- noise prevents copying
        self.decoder = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 28 * 28), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
den = DenoisingAE().to(device)
assert den(fixed_x.to(device)).shape == fixed_x.shape, "DenoisingAE output must be (B, 1, 28, 28)"
print("✓ TODO 4 check passed.")
train(den, train_loader, epochs=10)
score('denoising AE', den)

# Show it denoise: corrupt test images by masking ~50% of pixels, then reconstruct.
mask = (torch.rand_like(fixed_x) > 0.5).float()
noisy = fixed_x * mask
den.eval()
with torch.no_grad():
    cleaned = den(noisy.to(device)).cpu()
fig, axs = plt.subplots(2, len(fixed_x), figsize=(1.4 * len(fixed_x), 3))
for k in range(len(fixed_x)):
    axs[0, k].imshow(noisy[k].squeeze(), cmap='gray'); axs[0, k].axis('off')
    axs[1, k].imshow(cleaned[k].squeeze(), cmap='gray'); axs[1, k].axis('off')
fig.suptitle('Denoising AE   (top: corrupted input,  bottom: reconstruction)'); plt.show()

## Part 6 — Variational AE (generative)

The encoder now outputs a **mean** and a **log-variance**; we sample a code and add a **KL** term that keeps codes near $\mathcal{N}(0, I)$ — which is what lets us *generate*.

<img src='lab9_slides/ae_slide31_vae_arch.png' width='640'>

### TODO 5 — the reparameterization trick

In [ ]:
CODINGS = 16

class VAE(nn.Module):
    def __init__(self, codings=CODINGS, hidden=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden), nn.ReLU(),
            nn.Linear(hidden, 2 * codings))        # -> mean & logvar
        self.decoder = nn.Sequential(
            nn.Linear(codings, hidden), nn.ReLU(),
            nn.Linear(hidden, 28 * 28), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)))
    def encode(self, x):
        return self.encoder(x).chunk(2, dim=-1)    # (mean, logvar)
    def sample_codings(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        return mean + torch.randn_like(std) * std  # reparameterization
    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.sample_codings(mean, logvar)
        return self.decoder(z), mean, logvar

<img src='lab9_slides/ae_slide32b_kl_intro.png' width='640'>

<img src='lab9_slides/ae_slide34_vae_loss.png' width='640'>

### TODO 6 — the KL term in the loss

In [ ]:
def vae_loss(out, x, kl_weight=1.0):
    recon_x, mean, logvar = out
    recon = F.mse_loss(recon_x, x)
    kl = -0.5 * torch.sum(1 + logvar - logvar.exp() - mean.square(), dim=-1)   # (batch,)
    return recon + kl_weight * kl.mean() / 784

In [ ]:
vae = VAE().to(device)
xb = fixed_x.to(device)
out = vae(xb)
assert out[0].shape == xb.shape, "decoder output must match input shape"
za = vae.sample_codings(*vae.encode(xb))
zb = vae.sample_codings(*vae.encode(xb))
assert not torch.allclose(za, zb), "two code samples must differ (randomness!)"
assert torch.isfinite(vae_loss(out, xb)), "loss must be finite"
print("✓ TODO 5 & 6 checks passed.")
train(vae, train_loader, epochs=10, loss_fn=vae_loss)
score('VAE', vae, vae=True)
show_recon(vae, title='VAE', vae=True)

### Generate brand-new garments

The decoder turns any code into an image — sample codes straight from the prior and decode.

<img src='lab9_slides/ae_slide37_generate.png' width='640'>

In [ ]:
@torch.no_grad()
def generate(model, n=16):
    model.eval()
    z = torch.randn(n, CODINGS, device=device)
    return model.decoder(z).cpu()

gen = generate(vae, 16)
fig, axs = plt.subplots(2, 8, figsize=(12, 3))
for ax, im in zip(axs.ravel(), gen):
    ax.imshow(im.squeeze(), cmap='gray'); ax.axis('off')
fig.suptitle('Brand-new garments sampled from z ~ N(0, I)'); plt.show()

## Part 7 — Compare and visualize

A scoreboard of reconstruction error, plus a t-SNE map of the stacked-AE codes (colored by a class label the AE never saw).

<img src='lab9_slides/ae_slide17_tsne.png' width='640'>

<img src='lab9_slides/ae_slide40_comparison.png' width='640'>

In [ ]:
print('Reconstruction MSE on test (lower = better):')
for k, v in sorted(scoreboard.items(), key=lambda kv: kv[1]):
    print(f'  {k:<14} {v:.5f}')
print()
print('The VAE usually has higher recon MSE -- it pays for the KL term, which buys a')
print('smooth, *generative* latent space. The plain AEs only reconstruct.')

# t-SNE of the stacked-AE codes, colored by class (the AE never saw labels)
from sklearn.manifold import TSNE
stacked.eval(); codes, labs = [], []
with torch.no_grad():
    for x, y in DataLoader(test_ds, batch_size=512):
        codes.append(stacked.encoder(x.to(device)).cpu().numpy()); labs.append(y.numpy())
        if sum(len(c) for c in codes) >= 2000:
            break
C = np.concatenate(codes)[:2000]; L = np.concatenate(labs)[:2000]
emb = TSNE(n_components=2, init='pca', random_state=0).fit_transform(C)
plt.figure(figsize=(6, 5))
sc = plt.scatter(emb[:, 0], emb[:, 1], c=L, cmap='tab10', s=8, alpha=0.6)
plt.title('t-SNE of stacked-AE codes (colored by class)')
plt.colorbar(sc, ticks=range(10)); plt.show()

## Exercises

Pick a couple and report what you find (a sentence and a figure each):

1. **Bottleneck size.** Re-run the stacked AE with code size 8, 32, and 128. Plot recon MSE vs code size. Where are diminishing returns?
2. **Tie the weights.** Make the stacked decoder reuse the encoder's weight (transposed). Does accuracy hold with half the decoder parameters?
3. **β-VAE.** Sweep `kl_weight` in {0.1, 1, 4}. How do the generated samples and the reconstructions trade off?
4. **Anomaly detection.** Treat one Fashion-MNIST class as 'normal', train an AE on it only, and flag the other classes by reconstruction error. Report the ROC-AUC.
5. **Conv denoising.** Combine Parts 4 and 5: add input noise to the conv AE and compare denoising quality to the MLP version.

---
### Wrap-up
Same skeleton every time — *encode, then decode* — with a different constraint: a bottleneck (undercomplete), depth (stacked), locality (conv), corruption (denoising), or a sampled, KL-regularized code (VAE). Only the last one can **generate**.

